# Section 07 lab — Three-run e-commerce pipeline

Run the section setup once, then complete the five project stages in order. The complete transformation logic remains visible in this notebook.


## Section setup — run once


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Stage 1 — Inspect the configured source files


In [ ]:
import json

with open('project/run_config.json', encoding='utf-8') as stream:
    configuration = json.load(stream)
configuration


## Stage 2 — Execute all three runs and create one report per run

This cell reads each source format, validates records, joins references, writes Parquet, writes an audit row and builds a report from the saved destination data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from IPython.display import HTML, display
from decimal import Decimal
import os

PROJECT = 'project'
transaction_schema = StructType([
    StructField('transaction_id', StringType()), StructField('transaction_date', StringType()),
    StructField('customer_id', StringType()), StructField('product_id', StringType()),
    StructField('quantity', StringType()), StructField('unit_price', StringType()),
    StructField('status', StringType())])
customer_schema = StructType([
    StructField('customer_id', StringType()), StructField('customer_name', StringType()),
    StructField('city', StringType())])
product_schema = StructType([
    StructField('product_id', StringType()), StructField('product_name', StringType()),
    StructField('category', StringType()), StructField('list_price', StringType())])
customers = spark.read.schema(customer_schema).option('header', True).csv(f'{PROJECT}/reference/customers.csv')
products = spark.read.schema(product_schema).json(f'{PROJECT}/reference/products.json')
runs = [
    {'run_id': 'run-001', 'format': 'csv', 'path': 'source/run-001/transactions.csv'},
    {'run_id': 'run-002', 'format': 'json', 'path': 'source/run-002/transactions.json'},
    {'run_id': 'run-003', 'format': 'tsv', 'path': 'source/run-003/transactions.tsv'},
]
all_metrics = {}
os.makedirs(f'{PROJECT}/reports', exist_ok=True)

for run in runs:
    run_id = run['run_id']
    source_path = f"{PROJECT}/{run['path']}"
    reader = spark.read.schema(transaction_schema)
    if run['format'] == 'csv':
        raw = reader.option('header', True).csv(source_path)
    elif run['format'] == 'tsv':
        raw = reader.option('header', True).option('sep', '\t').csv(source_path)
    else:
        raw = reader.json(source_path)
    raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
    unique = raw.dropDuplicates(['transaction_id'])
    parsed = (unique.withColumn('transaction_id', F.trim('transaction_id'))
              .withColumn('customer_id', F.trim('customer_id'))
              .withColumn('product_id', F.trim('product_id'))
              .withColumn('status', F.upper(F.trim('status')))
              .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
              .withColumn('qty', F.expr('try_cast(quantity as int)'))
              .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
    quality_reason = F.concat_ws('; ',
        F.when(F.col('transaction_id').isNull() | (F.length('transaction_id') == 0), 'missing transaction_id'),
        F.when(F.col('order_date').isNull(), 'invalid transaction_date'),
        F.when(F.col('customer_id').isNull() | (F.length('customer_id') == 0), 'missing customer_id'),
        F.when(F.col('product_id').isNull() | (F.length('product_id') == 0), 'missing product_id'),
        F.when(F.col('qty').isNull() | (F.col('qty') <= 0), 'quantity must be positive'),
        F.when(F.col('price').isNull() | (F.col('price') < 0), 'unit_price must be non-negative'),
        F.when(F.col('status').isNull() | ~F.col('status').isin('COMPLETED', 'CANCELLED', 'RETURNED'), 'unknown status'))
    classified = parsed.withColumn('reject_reason', quality_reason)
    quality_rejected = classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache()
    valid = (classified.filter(F.col('reject_reason') == '').drop('reject_reason')
             .withColumn('line_total', F.col('qty') * F.col('price')))
    completed = valid.filter(F.col('status') == 'COMPLETED')
    joined = (completed.join(F.broadcast(customers), 'customer_id', 'left')
              .join(F.broadcast(products), 'product_id', 'left'))
    reference_reason = F.concat_ws('; ',
        F.when(F.col('customer_name').isNull(), 'unknown customer_id'),
        F.when(F.col('product_name').isNull(), 'unknown product_id'))
    reference_rejected = (joined.withColumn('reject_reason', reference_reason)
                          .filter(F.col('reject_reason') != '')
                          .withColumn('reject_stage', F.lit('REFERENCE')).cache())
    accepted = (joined.filter(F.col('customer_name').isNotNull() & F.col('product_name').isNotNull())
                .withColumn('order_month', F.date_format('order_date', 'yyyy-MM')).cache())
    rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
    curated_path = f'{PROJECT}/destination/curated_sales/run_id={run_id}'
    rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
    audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
    accepted.drop('run_id').write.mode('overwrite').partitionBy('order_month').parquet(curated_path)
    rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
    revenue = accepted.agg(F.sum('line_total').alias('value')).first()['value'] or Decimal('0')
    metrics = {
        'run_id': run_id, 'raw_rows': raw.count(),
        'duplicate_rows': raw.count() - unique.count(), 'unique_rows': unique.count(),
        'quality_rejected_rows': quality_rejected.count(), 'valid_rows': valid.count(),
        'non_completed_rows': valid.count() - completed.count(),
        'reference_rejected_rows': reference_rejected.count(),
        'rejected_rows': quality_rejected.count() + reference_rejected.count(),
        'accepted_sales_rows': accepted.count(), 'accepted_revenue': f'{revenue:.2f}'}
    all_metrics[run_id] = metrics
    spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
    destination = spark.read.parquet(curated_path)
    category_rows = (destination.groupBy('category')
                     .agg(F.countDistinct('transaction_id').alias('orders'),
                          F.sum('qty').alias('units'), F.sum('line_total').alias('revenue'))
                     .orderBy('category').collect())
    largest = max([float(row['revenue']) for row in category_rows] or [1.0])
    bars = ''.join(f"<p><b>{row['category']}</b><br><span style='display:inline-block;background:#2878b5;height:18px;width:{float(row['revenue'])/largest*70:.1f}%'></span> {float(row['revenue']):,.2f}</p>" for row in category_rows)
    report_html = f"<h2>{run_id} destination report</h2><p>Accepted sales: <b>{metrics['accepted_sales_rows']}</b> &nbsp; Rejected: <b>{metrics['rejected_rows']}</b> &nbsp; Revenue: <b>{metrics['accepted_revenue']}</b></p>{bars}"
    with open(f'{PROJECT}/reports/{run_id}.html', 'w', encoding='utf-8') as stream:
        stream.write(report_html)
    display(HTML(report_html))
    print(run_id, metrics)
    for frame in (raw, quality_rejected, reference_rejected, accepted):
        frame.unpersist()

all_metrics


## Stage 3 — Validate the three audit results


In [ ]:
PROJECT = 'project'
with open('project/validation/expected_metrics.json', encoding='utf-8') as stream:
    expected = json.load(stream)
with open('project/run_config.json', encoding='utf-8') as stream:
    configured_runs = [row['run_id'] for row in json.load(stream)['runs']]
for run_id in configured_runs:
    audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
    actual = spark.read.parquet(audit_path).first().asDict()
    for key, value in expected[run_id].items():
        assert actual[key] == value, (run_id, key, actual[key], value)
print('PASS: all three run audits match the expected results')


## Stage 4 — Build the combined report from destination Parquet


In [ ]:
from pyspark.sql import functions as F
from IPython.display import HTML, display
import os

PROJECT = 'project'
curated_base = f'{PROJECT}/destination/curated_sales'
destination = spark.read.option('basePath', curated_base).parquet(curated_base)
category_rows = (destination.groupBy('category')
                 .agg(F.countDistinct('transaction_id').alias('orders'),
                      F.sum('qty').alias('units'), F.sum('line_total').alias('revenue'))
                 .orderBy('category').collect())
total_revenue = destination.agg(F.sum('line_total')).first()[0]
largest = max([float(row['revenue']) for row in category_rows] or [1.0])
bars = ''.join(f"<p><b>{row['category']}</b><br><span style='display:inline-block;background:#2a9d8f;height:18px;width:{float(row['revenue'])/largest*70:.1f}%'></span> {float(row['revenue']):,.2f}</p>" for row in category_rows)
report_html = f"<h2>All-runs destination report</h2><p>Accepted sales: <b>{destination.count()}</b> &nbsp; Revenue: <b>{float(total_revenue):,.2f}</b></p>{bars}"
os.makedirs(f'{PROJECT}/reports', exist_ok=True)
with open(f'{PROJECT}/reports/all-runs.html', 'w', encoding='utf-8') as stream:
    stream.write(report_html)
display(HTML(report_html))
destination.groupBy('run_id', 'order_month').count().orderBy('run_id').show()
assert destination.count() == 21
assert f'{total_revenue:.2f}' == '103813.00'


## Stage 5 — Inspect the final aggregation plan


In [ ]:
from pyspark.sql import functions as F

curated_base = 'project/destination/curated_sales'
destination = spark.read.option('basePath', curated_base).parquet(curated_base)
destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).explain('formatted')
